# 05 — Graph Schema & Ontology (Milestone M3)

**DSML stage:** modeling (part 1). Defines the property-graph ontology from the feasibility studies and
applies it to Neo4j. Emits `artifacts/schema.cypher` so the schema is a versioned artifact, not notebook state.

## Ontology (MVP subset)

**Nodes** — `Company(cik, ticker, name, tier)` · `Filing(accession_no, form, filing_date, url)` ·
`FilingSection(section_key, section_id, title)` · `RiskFactor(risk_id, summary, category, embedding)` ·
`Metric(metric_id, metric, concept, value, unit, period_start, period_end)` · `Product(name, type)` ·
`ExportControl(rule_id, title, date)` *(Phase 5)* · `EvidenceSpan(chunk_id, text, char_start, char_end, source_url, embedding)`

**Edges** — `FILED` · `HAS_SECTION` · `DISCLOSES_RISK` · `SUPPLIES_TO` · `DEPENDS_ON` · `CUSTOMER_OF` ·
`COMPETES_WITH` · `AFFECTED_BY` · `REPORTS_METRIC` · `HAS_EVIDENCE` · `MENTIONS`

**Bitemporal pattern** (notebook 13 operates it): mutating edges carry `start_date`, `end_date` (null = open),
`status` (`Active` / `Deleted`). Time-travel = filter on those properties.

> Naming: properties are `snake_case` in the DB (the docs' `StartDate` → `start_date`).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
ARTIFACTS = PROJECT_ROOT / "artifacts"

# --- prerequisite guard ---
try:
    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
    )
    driver.verify_connectivity()
except Exception as e:
    raise RuntimeError(
        "Neo4j is not reachable. Install Neo4j Desktop (https://neo4j.com/download/), create & START a\n"
        "local DBMS (5.x), and put its password in .env as NEO4J_PASSWORD. See README 'Setup'."
    ) from e
print("Neo4j reachable —", driver.get_server_info().agent)

## 1. Schema DDL → `artifacts/schema.cypher`

All statements are idempotent (`IF NOT EXISTS`) so re-running is safe. Vector indexes are dimensioned for
`bge-large-en-v1.5` (1024, cosine).

In [ ]:
EMBED_DIM = 1024

SCHEMA_STATEMENTS = [
    # --- uniqueness constraints (also create backing indexes) ---
    "CREATE CONSTRAINT company_cik IF NOT EXISTS FOR (c:Company) REQUIRE c.cik IS UNIQUE",
    "CREATE CONSTRAINT company_name IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT filing_accession IF NOT EXISTS FOR (f:Filing) REQUIRE f.accession_no IS UNIQUE",
    "CREATE CONSTRAINT section_key IF NOT EXISTS FOR (s:FilingSection) REQUIRE s.section_key IS UNIQUE",
    "CREATE CONSTRAINT metric_id IF NOT EXISTS FOR (m:Metric) REQUIRE m.metric_id IS UNIQUE",
    "CREATE CONSTRAINT risk_id IF NOT EXISTS FOR (r:RiskFactor) REQUIRE r.risk_id IS UNIQUE",
    "CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (e:EvidenceSpan) REQUIRE e.chunk_id IS UNIQUE",
    "CREATE CONSTRAINT product_name IF NOT EXISTS FOR (p:Product) REQUIRE p.name IS UNIQUE",
    "CREATE CONSTRAINT exportcontrol_rule IF NOT EXISTS FOR (x:ExportControl) REQUIRE x.rule_id IS UNIQUE",
    # --- lookup indexes ---
    "CREATE INDEX company_ticker IF NOT EXISTS FOR (c:Company) ON (c.ticker)",
    "CREATE INDEX filing_date IF NOT EXISTS FOR (f:Filing) ON (f.filing_date)",
    "CREATE INDEX metric_period IF NOT EXISTS FOR (m:Metric) ON (m.period_end)",
    # --- temporal indexes on bitemporal edges ---
    "CREATE INDEX supplies_temporal IF NOT EXISTS FOR ()-[r:SUPPLIES_TO]-() ON (r.start_date, r.end_date, r.status)",
    "CREATE INDEX depends_temporal IF NOT EXISTS FOR ()-[r:DEPENDS_ON]-() ON (r.start_date, r.end_date, r.status)",
    "CREATE INDEX discloses_temporal IF NOT EXISTS FOR ()-[r:DISCLOSES_RISK]-() ON (r.start_date, r.end_date, r.status)",
    "CREATE INDEX affected_temporal IF NOT EXISTS FOR ()-[r:AFFECTED_BY]-() ON (r.start_date, r.end_date, r.status)",
    # --- vector indexes (bge-large-en-v1.5: 1024-dim, cosine) ---
    f"""CREATE VECTOR INDEX evidence_embedding IF NOT EXISTS FOR (e:EvidenceSpan) ON (e.embedding)
    OPTIONS {{indexConfig: {{`vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine'}}}}""",
    f"""CREATE VECTOR INDEX risk_embedding IF NOT EXISTS FOR (r:RiskFactor) ON (r.embedding)
    OPTIONS {{indexConfig: {{`vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine'}}}}""",
]

schema_path = ARTIFACTS / "schema.cypher"
schema_path.write_text(";\n\n".join(s.strip() for s in SCHEMA_STATEMENTS) + ";\n", encoding="utf-8")
print(f"{len(SCHEMA_STATEMENTS)} statements → {schema_path.relative_to(PROJECT_ROOT)}")

In [ ]:
with driver.session() as session:
    for stmt in SCHEMA_STATEMENTS:
        session.run(stmt)
print("Schema applied (idempotent)")

In [ ]:
# --- M3 (schema) assertion cell ---
with driver.session() as session:
    constraints = [r["name"] for r in session.run("SHOW CONSTRAINTS")]
    indexes = {r["name"]: r["type"] for r in session.run("SHOW INDEXES")}
expected_constraints = {"company_cik", "filing_accession", "section_key", "metric_id", "risk_id", "chunk_id"}
assert expected_constraints.issubset(set(constraints)), f"missing constraints: {expected_constraints - set(constraints)}"
assert indexes.get("evidence_embedding") == "VECTOR" and indexes.get("risk_embedding") == "VECTOR"
driver.close()
print(f"Schema OK — {len(constraints)} constraints, {len(indexes)} indexes (2 vector)")